## Startup-readiness-audit MVP 
### Purely using Gemini 2.5 flash, it seems to be decent.
I chose not to use a pdf text parser in case they had pictures with figures or graphics.

In [2]:
import google.genai as genai
import fitz  # PyMuPDF
import json
import os
from pydantic import BaseModel, Field
from typing import Optional, List
from dotenv import load_dotenv

In [3]:
load_dotenv()
client = genai.Client()

In [5]:
from pydantic import BaseModel, Field
from typing import Optional, List

class GradedSection(BaseModel):
    is_present: bool = Field(
        description="True if this topic is addressed anywhere in the deck. False if completely missing."
    )
    score: Optional[int] = Field(
        default=None, 
        description="Grade from 1 to 10 on clarity, strength, and VC-readiness. Null if is_present is False."
    )
    feedback: Optional[str] = Field(
        default=None, 
        description="Teacher's feedback: What made this strong? What was missing or confusing? Null if is_present is False."
    )
    evidence: Optional[str] = Field(
        default=None, 
        description="A verbatim quote or specific slide reference that justifies the score. Null if is_present is False."
    )

In [6]:
class KPIMetric(BaseModel):
    kpi_name: str = Field(description="Name of the metric (e.g., CAC, LTV, ARR)")
    kpi_value: str = Field(description="The numeric value or state")
    provenance: Optional[str] = Field(default=None, description="Verbatim quote backing up this KPI.")

class PitchDeckEvaluation(BaseModel):
    company_name: Optional[str] = Field(default=None)
    
    # --- The Core 10 Grading Rubric ---
    s1_problem: GradedSection = Field(description="Is the problem acute, clearly defined, and validated?")
    s2_solution: GradedSection = Field(description="Does the product actually solve the problem elegantly? Is the value prop clear?")
    s3_market_size: GradedSection = Field(description="Is the Total Addressable Market (TAM) large enough for venture scale, and logically calculated?")
    s4_product_and_tech: GradedSection = Field(description="Is the underlying magic, technology, or IP defensible and clear?")
    s5_business_model: GradedSection = Field(description="Is it clear how the company makes money (pricing, revenue streams)?")
    s6_go_to_market: GradedSection = Field(description="Do they have a realistic, scalable strategy to acquire customers?")
    s7_competition: GradedSection = Field(description="Do they understand their rivals and clearly define their competitive advantage?")
    s8_team: GradedSection = Field(description="Does the founding team have the right domain expertise, technical skills, or prior exits?")
    s9_traction_and_kpis: GradedSection = Field(description="Is there proof of concept (revenue, pilots, user growth)?")
    s10_the_ask_and_financials: GradedSection = Field(description="Is the fundraise amount clear? Are the financial projections realistic and tied to use of funds?")
    
    # --- Extras for the VC Hunter Scope ---
    extracted_kpis: list[KPIMetric] = Field(
        default_factory=list, 
        description="List of hard KPIs found. Empty list if none."
    )
    red_flags: list[str] = Field(
        default_factory=list, 
        description="List any unrealistic projections, glaring omissions, or concerning statements."
    )
    final_grade: Optional[str] = Field(
        default=None,
        description="Overall letter grade (A+, A, B, C, D, F) based on the sum of the parts."
    )

In [7]:
def calculate_fundability_score(evaluation: PitchDeckEvaluation) -> float:
    # Calculates a weighted fundability score out of 100 based on the Core 10 rubric.
    # Missing sections (is_present=False) receive a 0 for that category.
    
    # Define the weights (Must sum to 1.0)
    weights = {
        "s1_problem": 0.10,                 # 10%
        "s2_solution": 0.10,                # 10%
        "s3_market_size": 0.10,             # 10%
        "s4_product_and_tech": 0.10,        # 10%
        "s5_business_model": 0.05,          # 5%
        "s6_go_to_market": 0.05,            # 5%
        "s7_competition": 0.05,             # 5%
        "s8_team": 0.20,                    # 20% (Highly weighted)
        "s9_traction_and_kpis": 0.15,       # 15% (Highly weighted)
        "s10_the_ask_and_financials": 0.10  # 10%
    }
    
    total_score = 0.0
    
    # Iterate through each of the Core 10 sections in the evaluation
    for section_name, weight in weights.items():
        # Retrieve the GradedSection object dynamically using getattr
        section_data = getattr(evaluation, section_name)
        
        # If the section is present and has a score, calculate its weighted value
        if section_data.is_present and section_data.score is not None:
            # Score is out of 10. Multiply by 10 to make it out of 100, then apply weight.
            weighted_value = (section_data.score * 10) * weight
            total_score += weighted_value
            
    return round(total_score, 2)

In [12]:
def evaluate_pitch_deck(pdf_path: str) -> PitchDeckEvaluation:
    print(f"Uploading {pdf_path} to Gemini...")
    
    # 1. Upload the file securely to Google's servers for processing
    uploaded_file = client.files.upload(file=pdf_path)
    print(f"File uploaded successfully: {uploaded_file.name}")
    
    prompt = """
    You are an expert Venture Capital evaluator and a strict grader. 
    You are evaluating a startup pitch deck submitted as a final assignment. 
    Your task is to grade the deck based on the 'Core 10' startup principles.

    GRADING RUBRIC INSTRUCTIONS:
    1. For each of the 10 sections, determine if the startup actually addressed the topic (`is_present`).
    2. If present, assign a `score` from 1 to 10. Be highly critical. A 10 means it is top-tier Sequoia/Y-Combinator quality. A 5 means it is average and needs work.
    3. Provide constructive `feedback` as if speaking to the founder. Point out exactly what is strong and what is dangerously vague.
    4. Quote the deck directly in the `evidence` field to justify your grade.
    5. If a section is entirely missing from the slides, set `is_present` to false, and leave the score, feedback, and evidence as `null`.

    Do not grade them on information that is not in the deck. If they forgot their Business Model, fail that section.
    """

    print("Evaluating deck...")
    
    # 2. Pass BOTH the uploaded file object and the text prompt in a list
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[uploaded_file, prompt], 
        config={
            "response_mime_type": "application/json",
            "response_schema": PitchDeckEvaluation,
            "temperature": 0.0 
        }
    )
    
    # Optional: Clean up the file from Google's servers after you're done
    client.files.delete(name=uploaded_file.name)
    
    return response.parsed

In [17]:
def print_detailed_feedback(evaluation):
    print("=" * 60)
    print(f"--- DETAILED FEEDBACK FOR: {evaluation.company_name or 'Unknown Startup'} ---")
    print("=" * 60 + "\n")
    
    # A dictionary mapping your Pydantic attributes to readable headers
    core_10_sections = {
        "s1_problem": "1. Problem Statement",
        "s2_solution": "2. Solution & Value Prop",
        "s3_market_size": "3. Market Size (TAM)",
        "s4_product_and_tech": "4. Product & Technology",
        "s5_business_model": "5. Business Model",
        "s6_go_to_market": "6. Go-To-Market Strategy",
        "s7_competition": "7. Competition",
        "s8_team": "8. Team",
        "s9_traction_and_kpis": "9. Traction",
        "s10_the_ask_and_financials": "10. The Ask & Financials"
    }
    
    # Loop through the Core 10
    for attr_name, title in core_10_sections.items():
        # Fetch the GradedSection object dynamically
        section = getattr(evaluation, attr_name)
        
        print(title)
        if section.is_present:
            print(f"  Score:    {section.score}/10")
            print(f"  Feedback: {section.feedback}")
            print(f"  Evidence: \"{section.evidence}\"")
        else:
            print("  [!] MISSING: This section was not found in the deck.")
        print("-" * 50)
        
    # Print out the hard KPIs
    print("\n" + "=" * 60)
    print("EXTRACTED INDUSTRY KPIs")
    print("=" * 60)
    if evaluation.extracted_kpis:
        for kpi in evaluation.extracted_kpis:
            print(f"  • {kpi.kpi_name}: {kpi.kpi_value}")
            print(f"    (Provenance: \"{kpi.provenance}\")")
    else:
        print("  No specific KPIs were explicitly stated.")

    # Print out the Red Flags
    print("\n" + "=" * 60)
    print("DETECTED RED FLAGS")
    print("=" * 60)
    if evaluation.red_flags:
        for flag in evaluation.red_flags:
            print(f"  [X] {flag}")
    else:
        print("  None detected. Looking good!")

In [14]:
if __name__ == "__main__":
    sample_pdf = "sample_startup_deck.pdf" 
    
    if os.path.exists(sample_pdf):
        print("Starting evaluation pipeline...")
        
        # 1. LLM extracts and grades the individual sections
        evaluation = evaluate_pitch_deck(sample_pdf)
        
        # 2. Python calculates the weighted total score
        final_score = calculate_fundability_score(evaluation)
        
        # 3. Output the results
        print("\n" + "="*40)
        print(f"COMPANY: {evaluation.company_name or 'Unknown'}")
        print(f"FUNDABILITY SCORE: {final_score} / 100")
        print("="*40)
        
        # Example of how you can extract specific feedback for the dashboard
        if evaluation.s8_team.is_present:
            print(f"\nTeam Score: {evaluation.s8_team.score}/10")
            print(f"Team Feedback: {evaluation.s8_team.feedback}")
        else:
            print("\nCRITICAL WARNING: No Team slide found. Automatic 0 for this section.")
            
    else:
        print(f"Please place a '{sample_pdf}' in your notebook directory.")

Starting evaluation pipeline...
Uploading sample_startup_deck.pdf to Gemini...
File uploaded successfully: files/8rpbz70evupx
Evaluating deck...

COMPANY: 1906
FUNDABILITY SCORE: 83.0 / 100

Team Score: 10/10
Team Feedback: Your team is truly world-class. The founders and key personnel bring a highly relevant and impressive array of experience in finance, manufacturing, policy, marketing, sales, and scientific formulation within the cannabis and related industries. The specific achievements and past roles of each member demonstrate a strong capability to execute on your vision.


In [16]:
print_detailed_feedback(evaluation)

--- DETAILED FEEDBACK FOR: 1906 ---

1. Problem Statement
  Score:    9/10
  Feedback: You've done an excellent job articulating the problem. You clearly highlight the shortcomings of traditional cannabis consumption (smoking) and existing edibles, and effectively identify a significant unmet need for functional, discreet, and health-focused cannabis products. The supporting data from the Harris Poll and consumer quotes strongly validate your problem statement.
  Evidence: "CANNABIS AND SMOKING ARE ON A COLLISION COURSE (Slide 3); EDIBLES ARE THE FUTURE (Slide 4); 75% OF PEOPLE ARE INTERESTED IN CANNABIS AS MEDICINE (Slide 8); 'HEALTH' TRUMPS HIGH (Slide 29)"
--------------------------------------------------
2. Solution & Value Prop
  Score:    9/10
  Feedback: Your solution is exceptionally well-defined and directly addresses the problems you've identified. The emphasis on functional, fast-acting, great-tasting, discreet, and targeted cannabis products, along with the unique integrat